**TEST: VICOCKTAIL DATASET - GREEDY BASELINE**

In [81]:
import os
import sys

PROJECT_DIR = os.path.dirname(os.getcwd())

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

In [82]:
import yaml
import torch
import torch.nn as nn

from srcs.datasets.vicocktail import Collator, load_vicocktail
from srcs.nets.e2e import get_model
from srcs.trainer.trainer import HFTrainer, build_metric_fn, preprocess_logits_for_metrics
from srcs.spm.text_transofm import TextTransform
from transformers import TrainingArguments

with open(os.path.join(PROJECT_DIR, "config.yaml"), "r", encoding="utf-8") as f:
    configs = yaml.safe_load(f)

TEST_SIZE = 1.0
BASELINE_CHECKPOINT = os.path.join(PROJECT_DIR,"checkpoints","baseline","checkpoint-305877")
SAVE_DIR = os.path.join(PROJECT_DIR, 'experiments', "results")
SAMPLE_COUNT = 1

os.makedirs(SAVE_DIR, exist_ok=True)

In [83]:
ds = load_vicocktail(test_fraction=TEST_SIZE, splits=("test",))
text_transform = TextTransform()
test_collator = Collator(text_transform, "test")
ds

DatasetDict({
    test: Dataset({
        features: ['label', 'length', 'sample_id', 'video', 'video_length'],
        num_rows: 1167
    })
})

In [84]:
vsr_model = get_model(
        "baseline",
        text_transform.vocab_size,
        checkpoint_path=BASELINE_CHECKPOINT,
        **configs["model"],
    )

In [85]:
class Refiner(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=256,
        heads=4,
        linear_units=512,
        num_blocks=1,
        dropout=0.1,
        use_visual=False,
    ):
        super().__init__()
        self.use_visual = use_visual
        self.posterior_proj = nn.Sequential(
            nn.Linear(vocab_size, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        if use_visual:
            self.visual_norm = nn.LayerNorm(dim)
            self.h2_norm = nn.LayerNorm(dim)
            self.fuse = nn.Sequential(
                nn.Linear(dim * 3, dim),
                nn.LayerNorm(dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )

        self.encoder = ConformerEncoder(
            attention_dim=dim,
            attention_heads=heads,
            linear_units=linear_units,
            num_blocks=num_blocks,
            dropout_rate=dropout,
            positional_dropout_rate=dropout,
            attention_dropout_rate=0.0,
            cnn_module_kernel=31,
        )
        self.delta_head = nn.Linear(dim, vocab_size)
        nn.init.zeros_(self.delta_head.weight)
        nn.init.zeros_(self.delta_head.bias)

    def forward(self, posterior, mask, f_visual=None, h2=None):
        hidden = self.posterior_proj(posterior)

        if self.use_visual:
            if f_visual is None or h2 is None:
                raise ValueError("Visual contexts are required.")

            hidden = self.fuse(
                torch.cat(
                    [hidden, self.visual_norm(f_visual), self.h2_norm(h2)],
                    dim=-1,
                )
            )

        hidden = self.encoder(hidden, mask)[0]
        return self.delta_head(hidden)

In [86]:
evaluation_args = TrainingArguments(
    output_dir=SAVE_DIR,
    label_names=["labels", "label_lengths"],
    per_device_eval_batch_size=configs["evaluation"]["batch_size"],
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_num_workers=configs["evaluation"]["num_workers"],
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
)

In [87]:
# trainer = HFTrainer(
#     model=model,
#     args=evaluation_args,
#     eval_dataset=ds["test"],
#     data_collator=test_collator,
#     validation_collator=test_collator,
#     compute_metrics=build_metric_fn(text_transform),
#     preprocess_logits_for_metrics=preprocess_logits_for_metrics,
# )
# metrics = trainer.evaluate(metric_key_prefix="test")
# trainer.log_metrics("greedy_CTC_baseline", metrics)
# trainer.save_metrics("greedy_CTC_baseline", metrics)

In [88]:
sample_items = [ds["test"][index] for index in range(SAMPLE_COUNT)]
sample_batch = test_collator(sample_items)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = vsr_model.to(device).eval()

with torch.inference_mode():
    predictions = model.get_contexts(
        sample_batch["videos"].to(device),
        sample_batch["video_lengths"].to(device),
    )

In [89]:
predictions

{'loss': None,
 'proj_encoder': tensor([[[-0.1817,  0.2376,  0.2111,  ..., -0.0369, -0.0626,  0.5119],
          [-0.1589,  0.2769,  0.3226,  ..., -0.1099, -0.0913,  0.2399],
          [ 0.0406, -0.1123, -0.1234,  ..., -0.0079,  0.0345,  0.4416],
          ...,
          [-0.3288,  0.3764,  0.2225,  ...,  0.1204,  0.1768,  0.2648],
          [-0.3167,  0.6704,  0.2638,  ..., -0.1480,  0.1191,  0.3021],
          [-0.1216,  0.4416,  0.1163,  ...,  0.0320, -0.0473,  0.4635]]],
        device='cuda:0'),
 'logits': tensor([[[ 12.5506, -25.9546,  -0.7073,  ..., -27.3007, -21.3874, -26.6799],
          [ 12.0790, -25.4132,  -0.2810,  ..., -25.9580, -22.1441, -26.6643],
          [  6.5301, -19.1579,   0.4083,  ..., -18.8600, -15.6693, -19.7248],
          ...,
          [ 10.6637, -22.5062,   0.3108,  ..., -22.1028, -18.7082, -23.5469],
          [  7.5661, -18.2602,  -0.5322,  ..., -17.7974, -14.3634, -18.7946],
          [ 12.1941, -24.3745,   1.7551,  ..., -23.3790, -19.1240, -24.6014]]],

In [90]:
f_visual = predictions['proj_encoder']
logits = predictions['logits']

In [91]:
f_visual.size()

torch.Size([1, 206, 256])

In [92]:
logits.size()

torch.Size([1, 206, 3002])

In [93]:
ctc_posteriors = torch.softmax(logits, dim=-1, dtype=torch.float32)
ctc_posteriors.shape

torch.Size([1, 206, 3002])

In [94]:
from srcs.nets.utils import ctc_decode

token_ids = ctc_decode(
    ctc_posteriors,
    predictions["input_lengths"],
    text_transform.blank_id,
)
decoded_text = [text_transform.decode(ids) for ids in token_ids]
decoded_text

['khi mà mình có được những điều mà mình mới có thể cái một sống cái bỏ con rồi biết đứa ra đều đang đó làm kiếm kỹ làm để xác rất là đáng khó được không học ra']

In [95]:
def split_blank(self, logits):
    """
    Args:
        logits: [B,T,V]

    Returns:
        p_blank:
            [B,T,1]

        p_nonblank:
            [B,T,V-1]
            Conditional token distribution given non-blank.
    """

    probs = logits.softmax(dim=-1)

    p_blank = probs[
        ...,
        self.blank_id : self.blank_id + 1,
    ]

    nonblank_logits = torch.cat(
        [
            logits[..., :self.blank_id],
            logits[..., self.blank_id + 1:],
        ],
        dim=-1,
    )

    p_nonblank = nonblank_logits.softmax(dim=-1)

    return p_blank, p_nonblank